# 🇮🇳 Task 1 Preprocessing & Feature Engineering (iAWE)

This notebook processes the **iAWE** dataset for the NovaLume probationary task. 
It safely aligns the timestamps, downsamples to 6-second intervals, calculates 9 advanced features (**P, Q, Vrms, Irms, Delta P, Std, Mean, Time**), and feeds them directly to the GPU.

In [ ]:
import numpy as np
import pandas as pd
import tables
import tensorflow as tf
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')

# File Paths
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
H5_FILE_PATH = DATA_DIR / 'iawe.h5'

WINDOW_SIZE = 599
BATCH_SIZE = 1024
RESAMPLE_SEC = '6S' # Downsampling to 6-seconds

print("Libraries loaded.")

## 📊 1. Exploratory Data Analysis (EDA)

In [ ]:
def plot_raw_data(h5_path, aggregate_path, num_samples=10000):
    if not h5_path.exists(): return
        
    with tables.open_file(h5_path, mode='r') as f:
        table = f.get_node(f"{aggregate_path}/table")[:num_samples]
        active_power = table['values_block_0'][:, 0]
        reactive_power = table['values_block_0'][:, 1]
        voltage = table['values_block_0'][:, 4]
        current = table['values_block_0'][:, 6]
        
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
    ax1.plot(active_power, label='Active Power (P)', alpha=0.8, color='blue')
    ax1.plot(reactive_power, label='Reactive Power (Q)', alpha=0.7, color='orange')
    ax1.set_title('Raw Power Measurements')
    ax1.set_ylabel('Watts / VAR')
    ax1.legend()
    
    ax2.plot(voltage, label='Voltage (Vrms)', alpha=0.8, color='green')
    ax2.set_title('Raw Voltage Fluctuations')
    ax2.set_ylabel('Volts')
    ax2.set_xlabel('Time Steps')
    ax2.legend()
    plt.tight_layout()
    plt.show()

# plot_raw_data(H5_FILE_PATH, '/building1/elec/meter1')

In [ ]:
def plot_daily_energy_boxplot(h5_path):
    if not h5_path.exists(): return
    
    print("Aligning data for EDA...")
    with tables.open_file(h5_path, mode='r') as f:
        def load_series(path, name):
            data = f.get_node(path)[:] 
            dates = pd.to_datetime(data['index'], unit='ns').tz_localize('UTC').tz_convert('Asia/Kolkata')
            return pd.Series(data['values_block_0'][:, 0], index=dates, name=name)
            
        mains = load_series('/building1/elec/meter1/table', 'Mains')
        ac = load_series('/building1/elec/meter4/table', 'AC')
        wm = load_series('/building1/elec/meter6/table', 'Washing_Machine') # Meter 6 is Washing Machine
        
    # Align by Timestamp
    df_eda = pd.concat([mains, ac, wm], axis=1).resample('1min').mean().ffill().dropna()
    df_eda[df_eda < 0] = 0
    
    # Convert Watts to Daily kWh
    daily_energy = df_eda.resample('D').sum() * (1/60) / 1000
    
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=daily_energy)
    plt.title("Daily Energy Consumption Distribution (kWh)")
    plt.ylabel("Energy (kWh)")
    plt.tight_layout()
    plt.show()

# plot_daily_energy_boxplot(H5_FILE_PATH)

## ⚔️ 2. Aligned Feature Generator
Because appliances start recording at different times, we first align them in memory. We then dynamically engineer the required Statistical / Temporal features during batch generation.

In [ ]:
class IAWETask1Generator(tf.keras.utils.Sequence):
    def __init__(self, h5_path, aggregate_path, app1_path, app2_path, batch_size=1024, window_size=599, resample_sec='6S'):
        self.batch_size = batch_size
        self.window_size = window_size
        self.midpoint = window_size // 2
        
        print("Aligning and Downsampling Dataset...")
        with tables.open_file(h5_path, mode='r') as f:
            agg_data = f.get_node(f"{aggregate_path}/table")[:]
            dates = pd.to_datetime(agg_data['index'], unit='ns').tz_localize('UTC').tz_convert('Asia/Kolkata')
            
            df_agg = pd.DataFrame({
                'P': agg_data['values_block_0'][:, 0],
                'Q': agg_data['values_block_0'][:, 1],
                'Vrms': agg_data['values_block_0'][:, 4],
                'Irms': agg_data['values_block_0'][:, 6]
            }, index=dates)
            
            app1_data = f.get_node(f"{app1_path}/table")[:]
            app1_dates = pd.to_datetime(app1_data['index'], unit='ns').tz_localize('UTC').tz_convert('Asia/Kolkata')
            df_app1 = pd.Series(app1_data['values_block_0'][:, 0], index=app1_dates, name='App1')
            
            app2_data = f.get_node(f"{app2_path}/table")[:]
            app2_dates = pd.to_datetime(app2_data['index'], unit='ns').tz_localize('UTC').tz_convert('Asia/Kolkata')
            df_app2 = pd.Series(app2_data['values_block_0'][:, 0], index=app2_dates, name='App2')
            
        # Concatenate perfectly by timestamp and Downsample to 6-Seconds
        self.df = pd.concat([df_agg, df_app1, df_app2], axis=1).resample(resample_sec).mean().ffill().dropna()
        self.df[self.df < 0] = 0
        
        # Feature Engineering over the whole aligned dataset
        print("Engineering Features...")
        self.df['Delta_P'] = self.df['P'].diff().fillna(0)
        self.df['Rolling_Mean_P'] = self.df['P'].rolling(window=15, min_periods=1).mean()
        self.df['Rolling_Std_P'] = self.df['P'].rolling(window=15, min_periods=1).std().fillna(0)
        
        hours = self.df.index.hour + (self.df.index.minute / 60.0)
        self.df['Hour_Sin'] = np.sin(2 * np.pi * hours / 24)
        self.df['Hour_Cos'] = np.cos(2 * np.pi * hours / 24)
        
        # Normalize Inputs (Keep Targets in Watts)
        norm_cols = ['P', 'Q', 'Vrms', 'Irms', 'Delta_P', 'Rolling_Mean_P', 'Rolling_Std_P']
        for col in norm_cols:
            mean, std = self.df[col].mean(), self.df[col].std() + 1e-6
            self.df[col] = (self.df[col] - mean) / std
            
        # Fast Numpy Arrays
        feature_cols = ['P', 'Q', 'Vrms', 'Irms', 'Delta_P', 'Rolling_Mean_P', 'Rolling_Std_P', 'Hour_Sin', 'Hour_Cos']
        self.features_array = self.df[feature_cols].values
        self.app1_array = self.df['App1'].values
        self.app2_array = self.df['App2'].values
        
        self.total_windows = len(self.df) - self.window_size + 1
        print("Initialization Complete.")
        
    def __len__(self):
        return int(np.floor(self.total_windows / self.batch_size))
    
    def __getitem__(self, index):
        start_idx = index * self.batch_size
        
        X_batch, y_batch = [], []
        for i in range(start_idx, start_idx + self.batch_size):
            X_batch.append(self.features_array[i : i + self.window_size, :])
            y_batch.append([
                self.app1_array[i + self.midpoint],
                self.app2_array[i + self.midpoint]
            ])
            
        return np.array(X_batch), np.array(y_batch)

if H5_FILE_PATH.exists():
    gen = IAWETask1Generator(H5_FILE_PATH, '/building1/elec/meter1', '/building1/elec/meter4', '/building1/elec/meter6') # Meter 6 is WM
    print("Task 1 Feature Generator Ready.")

## 🔍 3. Visualizing Preprocessed Deep Learning Input

In [ ]:
def plot_preprocessed_window():
    if not H5_FILE_PATH.exists(): return
    
    X_batch, y_batch = gen[0]
    window = X_batch[0]
    
    labels = ['Active P', 'Reactive Q', 'Vrms', 'Irms', 'Delta P', 'Roll Mean', 'Roll Std', 'Hour Sin', 'Hour Cos']
    
    plt.figure(figsize=(15, 8))
    for i in range(4): 
        plt.plot(window[:, i], label=labels[i], alpha=0.7)
        
    plt.title(f"9-Channel Deep Learning Input Window (Shape: {window.shape})")
    plt.xlabel('Time Step (each step = 6 seconds)')
    plt.ylabel('Normalized Z-Score')
    plt.legend()
    plt.show()
    
    print(f"Target Labels [AC, Washing Machine]: {y_batch[0]} Watts")

# plot_preprocessed_window()